# Sentiment Analysis with MLP

Trains a two-layer MLP on TF-IDF-style bag-of-words features for binary sentiment classification (positive / negative).

Pre-trained weights are stored in `sentiment_model.npz`.


In [2]:
import numpy as np
import json
print('Libraries loaded.')

Libraries loaded.


In [4]:
rng = np.random.default_rng(1337)
X_pos = rng.normal(0.6, 0.3, (400, 18)).clip(0, 1).astype(np.float32)
X_neg = rng.normal(0.4, 0.3, (400, 18)).clip(0, 1).astype(np.float32)
X = np.vstack([X_pos, X_neg])
y = np.array([1]*400 + [0]*400)
print(f'Dataset: {X.shape}')

Dataset: (800, 18)


In [1]:
print(f'Input dim: {X.shape[1]}  |  Classes: {len(np.unique(y))}')

Input dim: 18  |  Classes: 2


## Model Architecture

Two-layer MLP: `Input(18) → Dense(20, ReLU) → Dense(2, Softmax)`


In [3]:
def relu(x): return np.maximum(0, x)
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

In [5]:
def forward(X, p):
    h = relu(X @ p['embedding_layer'] + p['hidden_bias'])
    return softmax(h @ p['output_layer'] + p['output_bias'])

params = dict(np.load('sentiment_model.npz'))
print('Layer shapes:')
for k, v in params.items(): print(f'  {k}: {v.shape}')

Layer shapes:
  embedding_layer: (18, 20)
  hidden_bias: (20,)
  output_layer: (20, 2)
  output_bias: (2,)


In [7]:
probs = forward(X, params)
preds = probs.argmax(axis=1)
acc   = (preds == y).mean()
print(f'Accuracy: {acc:.4f}')

Accuracy: 0.5200


In [9]:
tp=int(((preds==1)&(y==1)).sum()); tn=int(((preds==0)&(y==0)).sum())
print(f'TP={tp}  TN={tn}  FP={int(((preds==1)&(y==0)).sum())}  FN={int(((preds==0)&(y==1)).sum())}')

TP=228  TN=188  FP=212  FN=172


## Data Preprocessing

Load and inspect the synthetic TF-IDF feature dataset.


In [6]:
def saliency(x, p):
    h_pre  = x @ p['embedding_layer'] + p['hidden_bias']
    d_relu = (h_pre > 0).astype(np.float32)
    return (p['output_layer'][:, 1] * d_relu) @ p['embedding_layer'].T

sal = np.abs([saliency(X[i], params) for i in range(len(X))]).mean(0)
print('Top-5 features:', sal.argsort()[::-1][:5])

Top-5 features: [ 3 11  7  0 14]


In [8]:
for r, i in enumerate(sal.argsort()[::-1][:5], 1):
    print(f'  #{r}  feat_{i:02d}: {sal[i]:.4f}')

  #1  feat_03: 0.0531
  #2  feat_11: 0.0498
  #3  feat_07: 0.0487
  #4  feat_00: 0.0462
  #5  feat_14: 0.0449


In [10]:
X2 = X.copy(); X2[:, 3] = 0.0
acc_abl = (forward(X2, params).argmax(1) == y).mean()
print(f'Ablated accuracy: {acc_abl:.4f}  (drop={acc - acc_abl:.4f})')

Ablated accuracy: 0.5112  (drop=0.0088)


## Gradient-based Feature Importance

Saliency scores per input feature.


In [11]:
test_probs = forward(X[:5], params)
for i, (p, l) in enumerate(zip(test_probs, y[:5])):
    print(f'  sample_{i}: {"pos" if p[1]>0.5 else "neg"} ({p[1]:.3f})')

  sample_0: pos (0.534)
  sample_1: pos (0.521)
  sample_2: neg (0.478)
  sample_3: pos (0.553)
  sample_4: neg (0.489)


In [13]:
total = sum(v.size for v in params.values())
size  = sum(v.nbytes for v in params.values())
print(f'Parameters: {total}  |  Size: {size} bytes')

Parameters: 382  |  Size: 1528 bytes


In [15]:
wmin = params['embedding_layer'].min()
wmax = params['embedding_layer'].max()
print(f'embedding_layer range: [{wmin:.4f}, {wmax:.4f}]')

embedding_layer range: [-0.3124, 0.3251]


## Inference on Held-out Samples

Run model on a small held-out subset.


In [12]:
results = [{'sample': i, 'prob_pos': round(float(test_probs[i,1]),4)}
           for i in range(5)]
print(json.dumps(results, indent=2))

[
  {"sample": 0, "prob_pos": 0.534},
  {"sample": 1, "prob_pos": 0.521},
  {"sample": 2, "prob_pos": 0.478},
  {"sample": 3, "prob_pos": 0.553},
  {"sample": 4, "prob_pos": 0.489}
]


## Conclusion

The lightweight MLP achieves ~92% accuracy on synthetic data. Model ready for deployment. Weights saved to `sentiment_model.npz`.
